In [57]:
import logging
import warnings
import os
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from datetime import datetime
import hashlib

from dotenv import load_dotenv
import PyPDF2
import openai
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import re
import time

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

load_dotenv("../../.env", override=True)

True

## 1. Konfigurering og dataklassr

In [58]:
# Konfigurering
@dataclass
class EmbeddingConfig:
    """Konfigurering for embedding pipeline"""
    chunk_size: int = 1000  # Tokens per chunk
    chunk_overlap: int = 200  # Overlap mellom chunks
    embedding_model: str = "text-embedding-ada-002"  # OpenAI embedding model
    elasticsearch_index: str = "index_skatt_chunks"  # Ny index for chunked data
    batch_size: int = 100  # Batch size for embedding/ingest
    
@dataclass 
class DocumentChunk:
    """Representerer en chunk av et dokument"""
    text: str
    document_name: str
    chunk_index: int
    total_chunks: int
    metadata: Dict[str, Any]
    text_hash: str = None
    
    def __post_init__(self):
        if self.text_hash is None:
            self.text_hash = hashlib.md5(self.text.encode()).hexdigest()

config = EmbeddingConfig()
openai.api_key = os.environ.get("OPENAI_API_KEY")

es_client = Elasticsearch(
    ["https://elasticsearch-llm-spring-glitter-3589.fly.dev"],
    basic_auth=('elastic', os.environ.get("ELASTIC_PASSWORD")),
    verify_certs=False
)

print(f"Config: {config}")
print(f"ES health: {es_client.cluster.health()}")

Config: EmbeddingConfig(chunk_size=1000, chunk_overlap=200, embedding_model='text-embedding-ada-002', elasticsearch_index='index_skatt_chunks', batch_size=100)
ES health: {'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 5, 'active_shards': 5, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 5, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 50.0}


## 2. PDF Lesing og Tekst Ekstrahering

In [59]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Ekstraherer tekst fra PDF-fil
    
    Args:
        pdf_path: Path til PDF-fil
        
    Returns:
        Rå tekst fra PDF
    """
    text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num, page in enumerate(pdf_reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += f"\n\n--- Side {page_num + 1} ---\n\n{page_text}"
    
    # Enkel tekstopprydding
    text = re.sub(r'\n\s*\n\s*\n', '\n\n', text)  # Fjern overflødige linjeskift
    text = re.sub(r'\s+', ' ', text)  # Normaliser mellomrom
    text = text.strip()
    
    return text

## 3. Text Chunking med Overlapp

In [60]:
def estimate_token_count(text: str) -> int:
    """
    Estimerer antall tokens i tekst (ca. 4 karakterer per token for norsk)
    """
    return len(text) // 4

def chunk_text_with_overlap(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """
    Deler tekst opp i chunks med overlap
    
    Args:
        text: Tekst som skal deles opp
        chunk_size: Maks antall tokens per chunk
        chunk_overlap: Antall tokens som overlapper mellom chunks
        
    Returns:
        Liste med text chunks
    """
    # Konverter token-størrelser til karakter-estimat
    char_chunk_size = chunk_size * 4
    char_overlap = chunk_overlap * 4
    
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        # Beregn slutten av denne chunken
        end = start + char_chunk_size
        
        if end > text_length:
            end = text_length
        
        # Prøv å finne en naturlig pause (setning eller paragraf)
        if end < text_length:
            # Se etter punktum eller dobbelt linjeskift
            sentence_end = text.rfind('.', start, end)
            paragraph_end = text.rfind('\n\n', start, end)
            
            natural_end = max(sentence_end, paragraph_end)
            if natural_end > start + char_chunk_size // 2:  # Bare hvis vi finner noe rimelig
                end = natural_end + 1
        
        chunk = text[start:end].strip()
        if chunk:  # Bare legg til non-empty chunks
            chunks.append(chunk)
        
        # Beregn start for neste chunk (med overlap)
        if end >= text_length:
            break
        start = end - char_overlap
        
        # Sikre at vi ikke går bakover
        if start <= (chunks[-1].__len__() // 2 if chunks else 0):
            start = end - char_overlap // 2
    
    return chunks

def create_document_chunks(pdf_path: Path, config: EmbeddingConfig) -> List[DocumentChunk]:
    """
    Lager DocumentChunk objekter fra en PDF
    
    Args:
        pdf_path: Path til PDF
        config: EmbeddingConfig med chunking parametere
        
    Returns:
        Liste med DocumentChunk objekter
    """
    raw_text = extract_text_from_pdf(pdf_path)
    
    text_chunks = chunk_text_with_overlap(raw_text, config.chunk_size, config.chunk_overlap)
    
    document_name = pdf_path.stem
    
    document_chunks = []
    for i, chunk_content in enumerate(text_chunks):
        chunk = DocumentChunk(
            text=chunk_content,
            document_name=document_name,
            chunk_index=i,
            total_chunks=len(text_chunks),
            metadata={
                'source_file': pdf_path.name,
                'created_at': datetime.now().isoformat(),
                'estimated_tokens': estimate_token_count(chunk_content)
            }
        )
        document_chunks.append(chunk)
    
    return document_chunks


## 4. OpenAI Embedding Funksjonalitet

In [61]:
def embed_chunks(chunks: List[DocumentChunk]) -> List[DocumentChunk]:
    """
    Enkel embedding av chunks - en og en
    """
    embedded_chunks = []
    for i, chunk in enumerate(chunks):
        try:
            response = openai.embeddings.create(
                input=chunk.text,
                model=config.embedding_model
            )
            
            chunk.metadata['embedding'] = response.data[0].embedding
            chunk.metadata['embedding_model'] = config.embedding_model
            embedded_chunks.append(chunk)
            
        except Exception as e:
            print(f"❌ Feil på chunk {i}: {e}")
    
    return embedded_chunks

def create_es_index(index_name: str):
    """
    Enkel index-opprettelse
    """
    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)
        print(f"🗑️ Slettet gammel index: {index_name}")
    
    mapping = {
        "mappings": {
            "properties": {
                "content": {"type": "text"},
                "document_name": {"type": "keyword"},
                "chunk_index": {"type": "integer"},
                "embedding": {"type": "dense_vector", "dims": 1536}
            }
        }
    }
    
    es_client.indices.create(index=index_name, body=mapping)
    print(f"✅ Opprettet index: {index_name}")

def ingest_chunks_to_es(chunks: List[DocumentChunk], index_name: str):
    """
    Enkel ingest til Elasticsearch
    """
    count = 0
    for chunk in chunks:
        if 'embedding' in chunk.metadata:
            doc = {
                "content": chunk.text,
                "document_name": chunk.document_name,
                "chunk_index": chunk.chunk_index,
                "embedding": chunk.metadata['embedding']
            }
            
            es_client.index(index=index_name, body=doc)
            count += 1
    
    return count

## 5. Elasticsearch Index Ingest All Docs

In [62]:
def process_all_documents(pdf_files: List[Path], 
                          config: EmbeddingConfig) -> Dict[str, Any]:
    """
    Prosesserer alle PDF dokumenter gjennom hele pipeline
    
    Args:
        pdf_files: Liste med PDF filer å prosessere
        config: EmbeddingConfig
        es_client: Elasticsearch klient
        
    Returns:
        Dict med resultater fra prosesseringen
    """
    
    results = {
        "processed_files": [],
        "total_chunks": 0,
        "total_embedded": 0,
        "total_ingested": 0,
        "errors": [],
        "processing_time": 0
    }
    
    start_time = time.time()
    
    create_es_index(config.elasticsearch_index)
    
    all_chunks = []
    
    for pdf_file in pdf_files:
        chunks = create_document_chunks(pdf_file, config)
        all_chunks.extend(chunks)
        
        results["processed_files"].append({
            "file": pdf_file.name,
            "chunks": len(chunks)
        })
    
    results["total_chunks"] = len(all_chunks)
    
    total_batches = (len(all_chunks) + config.batch_size - 1) // config.batch_size
    print(f"📊 Prosesserer {len(all_chunks)} chunks i {total_batches} batcher (batch size: {config.batch_size})")

    for batch_num, i in enumerate(range(0, len(all_chunks), config.batch_size), 1):
        chunk_batch = all_chunks[i:i + config.batch_size]
        
        print(f"🚀 Batch {batch_num}/{total_batches}: Starter {len(chunk_batch)} chunks")
        batch_start = time.time()

        embedded_chunks = embed_chunks(chunk_batch)
        results["total_embedded"] += len(embedded_chunks)

        if embedded_chunks:
            ingested_count = ingest_chunks_to_es(
                embedded_chunks,
                config.elasticsearch_index
            )
            results["total_ingested"] += ingested_count
            
        batch_time = time.time() - batch_start
        remaining_batches = total_batches - batch_num
        est_remaining = batch_time * remaining_batches
        est_total = batch_time * total_batches
        print(f"✅ Batch {batch_num}/{total_batches} ferdig ({batch_time:.1f}s) - Est. remaining: {est_remaining/60:.1f}m | Est. total: {est_total/60:.1f}m\n")
    
    results["processing_time"] = time.time() - start_time
    
    return results

## 6. Ingest alle docs

In [63]:
# Konfigurer for produksjon (større batches, raskere prosessering)
production_config = EmbeddingConfig(
    chunk_size=800,  # Litt mindre chunks for bedre presisjon
    chunk_overlap=150,
    elasticsearch_index="skatt_chunks_2025", 
    batch_size=50  # Mindre batch for OpenAI rate limits
)

raw_data_path = Path("../../data/raw/")
pdf_files = list(raw_data_path.glob("*.pdf"))

print(f"Funnet {len(pdf_files)} PDF-filer:")
print(f"Filer: {pdf_files}")

results = process_all_documents(pdf_files, production_config)

print("\\n=== PROSESSERING KOMPLETT ===")
print(f"⏱️  Total tid: {results['processing_time']:.2f} sekunder")
print(f"📁 Prosesserte filer: {len(results['processed_files'])}")
print(f"📄 Totale chunks: {results['total_chunks']}")
print(f"🤖 Embedded chunks: {results['total_embedded']}")
print(f"📇 Ingesterte chunks: {results['total_ingested']}")

if results['errors']:
    print(f"❌ Feil: {len(results['errors'])}")
    for error in results['errors']:
        print(f"   - {error}")


Funnet 4 PDF-filer:
Filer: [PosixPath('../../data/raw/251013_merverdiavgiftsloven.pdf'), PosixPath('../../data/raw/251213_skatteloven.pdf'), PosixPath('../../data/raw/skatte-abc-2025.pdf'), PosixPath('../../data/raw/251013_skatteforvaltningsloven.pdf')]
🗑️ Slettet gammel index: skatt_chunks_2025
✅ Opprettet index: skatt_chunks_2025
📊 Prosesserer 3045 chunks i 61 batcher (batch size: 50)
🚀 Batch 1/61: Starter 50 chunks
✅ Batch 1/61 ferdig (22.9s) - Est. remaining: 22.9m | Est. total: 23.2m

🚀 Batch 2/61: Starter 50 chunks
✅ Batch 2/61 ferdig (16.9s) - Est. remaining: 16.6m | Est. total: 17.2m

🚀 Batch 3/61: Starter 50 chunks
✅ Batch 3/61 ferdig (15.8s) - Est. remaining: 15.3m | Est. total: 16.1m

🚀 Batch 4/61: Starter 50 chunks
✅ Batch 4/61 ferdig (15.9s) - Est. remaining: 15.1m | Est. total: 16.2m

🚀 Batch 5/61: Starter 50 chunks
✅ Batch 5/61 ferdig (15.8s) - Est. remaining: 14.7m | Est. total: 16.1m

🚀 Batch 6/61: Starter 50 chunks
✅ Batch 6/61 ferdig (17.4s) - Est. remaining: 15.9m |

# Test ES index

In [38]:
    
def test_vector_search(query_text: str, index_name: str, top_k: int = 5):
    """
    Test vector search mot din nye chunked index
    (samme tilnærming som i esSearchConsumer.ts)
    """
    print(f"🔍 Tester vector search: '{query_text}'")
    
    
    query_response = openai.embeddings.create(
        input=query_text,
        model=config.embedding_model
    )
    query_vector = query_response.data[0].embedding
    
    # 2. Vector search (samme struktur som searchMatchVector i esSearchConsumer.ts)
    print(f"  🎯 Søker i index: {index_name}")


    search_response = es_client.search(
    index = index_name,
    size = top_k,
    knn = {
            "field": "embedding",
            "query_vector": query_vector,
            "k": 20,
            "num_candidates": 100,
            "boost": 0.1,
        }
    )

    
    # 3. Vis resultater
    hits = search_response['hits']['hits']
    print(f"\\n✅ Fant {len(hits)} resultater:")
    print(f"Hits: {hits}")
    
    for i, hit in enumerate(hits):
        source = hit['_source']
        score = hit['_score']
        
        print(f"\\n{i+1}. {source['document_name']} (chunk {source['chunk_index']}) - Score: {score:.3f}")
        print(f"   Text: {source['content'][:200]}...")
    
    return hits

def test_keyword_search(query_text: str, index_name: str, top_k: int = 5):
    """
    Test keyword search (samme som searchMatchKeyword i esSearchConsumer.ts)
    """
    print(f"🔍 Tester keyword search: '{query_text}'")
    
    response = es_client.search(
      index=index_name,
      size=top_k,
      body={
          "query": {
              "match": {
                  "content": {
                      "query": query_text,
                      "boost": 0.9
                  }
              }
          }
      }
  )
    
    hits = response['hits']['hits']
    print(f"\\n✅ Keyword search fant {len(hits)} resultater:")
    
    for i, hit in enumerate(hits):
        source = hit['_source']
        score = hit['_score']
        
        print(f"\\n{i+1}. {source['document_name']} - Score: {score:.3f}")
        print(f"   Text: {source['content'][:150]}...")
    
    return hits
    



In [64]:
count = es_client.count(index="skatt_chunks_2025")["count"]
print(f"Chunks i ES: {count}")

Chunks i ES: 3045
